# A lightweight demo on how to use the model with 2D data (same procedure for 3D).
- ## Section 1: File structure
- ## Section 2: Inference based on trained model
- ## Section 3: Visualization and validation for results

# Section 1: File Structure

- **Dataset (`grain2d/`)**: 2D PMC dataset for training and validation (train/valid/test)
- **checkpoints (`checkpoints/`)**: trained model

# Section2: Inference based on trained model from checkpoints
- **Section 2.1: Inference based on GNN only**
- **Section 2.2: Inference based on AE + GNN (original)**
- **Section 2.3: Inference based on AE + GNN (latent)**

### section 2.1 inference based on GNN only

In [ ]:
import subprocess, sys, os


for f in [
    "grain2d/valid.npy",
]:
    print(("OK  " if os.path.exists(f) else "MISS"), f)


cmd = [
    sys.executable, "-m", "NPS.main",
    "--mode=predict",
    "--cpu",
    "--dir", f"checkpoints/grain_NPS-batch8_lr4e-3_nin1_nout1_noise1e-3_lossL2_nmp3_nhid96_nae1_nencdec3", 
    "--data=grain2d",
    "--dataset=longclip", "--frame_shape=64", "--nfeat_in=1",
    "--periodic", "--pointgroup=4m", "--dim=2",
    "--model=MeshGraphNets",         
    "--trainer=MeshGraphNets",       
    "--gnnmodel=NPS",
    "--autoencoder=rev2wae",
    "--feat_out_method=id",          
    "--act=silu", "--nfeat_hid=96",
    "--n_mpassing=3", "--nlayer_mlp_encdec=3",
    "--batch=8", "--lr=3e-3", "--n_in=1", "--n_out=1",
    "--noise_op=add_normal/0:1/1e-3",
    "--nepoch=50", "--epoch_size=2000",
    "--n_out_valid=19",
    "--loss=L2", "--scheduler=plateau",
    "--n_traj_out=300", "--n_out_predict=100",
    "--clip_step_valid=1",
]

env = dict(os.environ)
env["PYTHONPATH"] = env.get("PYTHONPATH", "") + ":" + os.getcwd()
env["CUDA_VISIBLE_DEVICES"] = "-1"

subprocess.run(cmd, env=env, check=True)

### section 2.2 inference based on AE + GNN (original)

In [ ]:
import subprocess, sys, os


for f in [
    "grain2d/valid.npy",
]:
    print(("OK  " if os.path.exists(f) else "MISS"), f)

for nae in [1, 2]:
    ae_stride = ",".join(map(str, [1, 2, 2, 2][:nae]))
    ae_block = ",".join(map(str, [2, 2, 2, 2][:nae]))

    cmd = [
        sys.executable, "-m", "NPS.main",
        "--mode=predict",
        "--cpu",
        "--dir", f"checkpoints/grain_NPS_autoencoder-batch8_lr4e-3_nin1_nout1_noise1e-3_lossL2_nmp3_nhid96_nae{nae}_nencdec3", 
        "--data=grain2d",
        "--dataset=longclip", "--frame_shape=64", "--nfeat_in=1",
        "--periodic", "--pointgroup=4m", "--dim=2",
        "--model=MeshGraphNets",         
        "--trainer=MeshGraphNets",       
        "--gnnmodel=NPS_autoencoder",
        "--autoencoder=rev2wae",
        "--feat_out_method=id",          
        "--act=silu", "--nfeat_hid=96",
        "--n_mpassing=3", "--nlayer_mlp_encdec=3",
        "--batch=8", "--lr=3e-3", "--n_in=1", "--n_out=1",
        "--noise_op=add_normal/0:1/1e-3",
        "--nepoch=50", "--epoch_size=2000",
        "--n_out_valid=19",
        "--loss=L2", "--scheduler=plateau",
        "--n_traj_out=300", "--n_out_predict=100",
        "--clip_step_valid=1",
        f"--nstrides_2wae={ae_stride}",
        f"--nblocks_2wae={ae_block}",
    ]

    env = dict(os.environ)
    env["PYTHONPATH"] = env.get("PYTHONPATH", "") + ":" + os.getcwd()
    env["CUDA_VISIBLE_DEVICES"] = "-1" 

    print(f"\nRunning nae={nae}:")
    print(" ".join(cmd))
    subprocess.run(cmd, env=env, check=True)

### section 2.3 inference based on AE + GNN (LATENT)

In [ ]:
import subprocess, sys, os


for f in [
    "grain2d/valid.npy",
]:
    print(("OK  " if os.path.exists(f) else "MISS"), f)

for nae in [1, 2]:
    ae_stride = ",".join(map(str, [1, 2, 2, 2][:nae]))
    ae_block = ",".join(map(str, [2, 2, 2, 2][:nae]))
    cmd = [
        sys.executable, "-m", "NPS.main",
        "--mode=predict",
        "--cpu",
        "--dir", f"checkpoints/grain_NPS_autoencoder-batch8_lr4e-3_nin1_nout1_noise1e-3_lossL2_nmp3_nhid96_nae{nae}_nencdec3", 
        "--data=grain2d",
        "--dataset=longclip", "--frame_shape=64", "--nfeat_in=1",
        "--periodic", "--pointgroup=4m", "--dim=2",
        "--model=MeshGraphNets",         
        "--trainer=MeshGraphNets",       
        "--gnnmodel=NPS_autoencoder",
        "--autoencoder=rev2wae",
        "--feat_out_method=id",          
        "--act=silu", "--nfeat_hid=96",
        "--n_mpassing=3", "--nlayer_mlp_encdec=3",
        "--batch=8", "--lr=3e-3", "--n_in=1", "--n_out=1",
        "--noise_op=add_normal/0:1/1e-3",
        "--nepoch=50", "--epoch_size=2000",
        "--n_out_valid=19", "--infer_mode=optimize",
        "--loss=L2", "--scheduler=plateau",
        "--n_traj_out=300", "--n_out_predict=100",
        "--clip_step_valid=1",
        f"--nstrides_2wae={ae_stride}",
        f"--nblocks_2wae={ae_block}",
    ]

    env = dict(os.environ)
    env["PYTHONPATH"] = env.get("PYTHONPATH", "") + ":" + os.getcwd()
    env["CUDA_VISIBLE_DEVICES"] = "-1" 

    print(f"\nRunning nae={nae}:")
    print(" ".join(cmd))
    subprocess.run(cmd, env=env, check=True)

# Section3: Visualization and Validation

### visualization based on the rollout result from section 2, AE+GNN (latent) is visualized only at the final step.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

gt = np.load('grain2d/valid.npy')              
pd_org = np.load('checkpoints/grain_NPS_autoencoder-batch8_lr4e-3_nin1_nout1_noise1e-3_lossL2_nmp3_nhid96_nae2_nencdec3/pd.npy')
pd_latent = np.load('checkpoints/grain_NPS_autoencoder-batch8_lr4e-3_nin1_nout1_noise1e-3_lossL2_nmp3_nhid96_nae2_nencdec3/pd_latent.npy')
pd_latent = pd_latent[0,1]   

gt = gt[..., 0]
pd_org = pd_org[..., 0]

pl = pd_latent
while pl.ndim > 2 and pl.shape[0] == 1:
    pl = np.squeeze(pl, axis=0)
if pl.ndim == 3 and pl.shape[-1] == 1:
    pl = pl[..., 0]
assert pl.ndim == 2, f"Unexpected pd_latent shape after squeeze: {pl.shape}"
pd_latent_oneframe = pl

sample_idx = 0
gt_sample = gt[sample_idx]     # (20, 64, 64)
pd_org_sample = pd_org[0]      # (100, 64, 64)
timesteps = [0, 1, 9, 19]
latent_t = 19

def draw_placeholder(ax, lw=0.6):

    rect = Rectangle((0.2, 0), 0.6, 1, transform=ax.transAxes,
                     fill=False, edgecolor='0.5', linewidth=lw)
    ax.add_patch(rect)
    ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(len(timesteps), 3, figsize=(9, 9))

for i, t in enumerate(timesteps):
    # GT
    axes[i, 0].imshow(gt_sample[t], cmap="gray", interpolation="nearest")
    axes[i, 0].set_title(f"GT t={t}")
    axes[i, 0].axis("off")

    # pd_org
    axes[i, 1].imshow(pd_org_sample[t], cmap="gray", interpolation="nearest")
    axes[i, 1].set_title(f"pd_org t={t}")
    axes[i, 1].axis("off")

    if t == latent_t:
        axes[i, 2].imshow(pd_latent_oneframe, cmap="gray", interpolation="nearest")
        axes[i, 2].set_title(f"pd_latent (t={latent_t})")
        axes[i, 2].axis("off")
    else:
        draw_placeholder(axes[i, 2], lw=0.8)
        axes[i, 2].set_title("pd_latent")
        axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

### validation with SSIM based on the rollout result from section2

In [ ]:
gt = np.load('grain2d/valid.npy')
gnn_only = np.load('checkpoints/grain_NPS-batch8_lr4e-3_nin1_nout1_noise1e-3_lossL2_nmp3_nhid96_nae1_nencdec3/pd.npy')
ae1 = np.load('checkpoints/grain_NPS_autoencoder-batch8_lr4e-3_nin1_nout1_noise1e-3_lossL2_nmp3_nhid96_nae1_nencdec3/pd.npy')
ae2 = np.load('checkpoints/grain_NPS_autoencoder-batch8_lr4e-3_nin1_nout1_noise1e-3_lossL2_nmp3_nhid96_nae2_nencdec3/pd.npy')


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))
rmse_all_gnn = []
rmse_all_ae1 = []
rmse_all_ae2 = []
rmse_all_ae3 = []
for i in range(20):
    rmse_all_gnn.append(rmse(gt[0,i],gnn_only[0,i]))
    rmse_all_ae1.append(rmse(gt[0,i],ae1[0,i]))
    rmse_all_ae2.append(rmse(gt[0,i],ae2[0,i]))

plt.rcParams['font.family'] = 'Nimbus Roman'
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(np.arange(20),rmse_all_gnn,label = 'GNN Baseline')


ax.plot(np.arange(20),rmse_all_ae1,label = 'Linear Compression Ratio 1')
ax.plot(np.arange(20),rmse_all_ae2,label = 'Linear Compression Ratio 2')
ax.set_ylabel('RMSE')
ax.set_xlabel('Simulation Step', fontsize=13)
ax.set_xticks(np.arange(0, 20, 5))
ax.grid(False)
fig.subplots_adjust(bottom=0.26)
ax.legend(loc='best', fontsize=12, frameon=False)
fig.text(0.5, 0.11, 'Figure 2(e): 2D RMSE comparison', ha='center', fontsize=13)
fig.savefig("rmse_ae2d.svg", format='svg',bbox_inches='tight',pad_inches=0.1)